<div align="right"><a href="S31-PSD-Estimation-using-Corelogram.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L31-Schaetzung-des-Leistungsdichtespektrums-nach-Blackman-Tuckey.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L30 Schätzung des Leistungsdichtespektrums mit Periodogramm](./L30-Schaetzung-des-Leistungsdichtespektrums-mit-Periodogramm.ipynb) | [Aufgabe 31](./A31-Schaetzung-des-Leistungsdichtespektrums-nach-Blackman-Tuckey.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L32 Schätzung des Leistungsdichtespektrums nach Bartlett und Welch](./L32-Schaetzung-des-Leistungsdichtespektrums-nach-Bartlett-und-Welch.ipynb)

</div>

---


### Aufgabe: Traditionelle Spektrumschätzung mit der Korrelationsmethode

Der Blackman–Tuckey-Algorithmus liefert eine Schätzung $\hat{S}{XX}^{BT}(\mathrm{e}^{j\Omega})$ des Leistungsdichtespektrums $S{XX}(\mathrm{e}^{j\Omega})$ basierend auf der Autokorrelation $r_{XX}(\kappa)$.

Dies umfasst:

* Eine **biasbehaftete Schätzung** der ACF-Reihe
  $\hat{r}_{XX}(\kappa)$ für $|\kappa| \leq M-1$
  unter Verwendung des Rader-Algorithmus,

* Eine **Multiplikation der ACF-Reihe** mit einem Fenster der Länge $2M-1$:

$$
\hat{r}_{XX}^{BT}(\kappa)
=========================

\hat{r}_{XX}(\kappa)\cdot f_M(\kappa)
$$

* Eine **DFT zur Berechnung des Leistungsdichtespektrums**:

$$
\hat{S}_{X_iX_i}^{BT}(\mathrm{e}^{j\Omega})
===========================================

\mathrm{DFT}{\hat{r}_{XX}^{BT}(\kappa)}
$$

Schätzen Sie die Leistungsdichtespektren beider Prozesse aus Aufgabe dsv8a02e mit dem Blackman–Tuckey-Algorithmus unter Verwendung eines rechteckigen oder dreieckigen Fensters für $M = 8$, $32$ und $128$.

Warum ist die Spektrumschätzung des MA-Modells bei $M = 8$ besonders gut?

---

**MATLAB-Hinweise:**

$rxx = lrader(M, x)$  Schätzung mit Rader-Algorithmus $triang(2*M-1)$   Dreiecks-(Bartlett)-Fenster der Länge $2*M-1$


In [ ]:
# BEGIN SOLUTION
% Modellrauschprozesse
N     = 2^12;           % Anzahl der Abtastwerte um das System in den eingeschw.
N_vor = 500;            % Zustand zu setzen

n  = randn(1,N+N_vor);  

h1 = [1 -4 6 -4 +1]; x1_all = filter(h1,1,n); x1=x1_all(N_vor+1:length(x1_all));
h2 = [1 0.8];        x2_all = filter(1,h2,n); x2=x2_all(N_vor+1:length(x2_all));

% "Wahres" Leistungsdichtespektrum zum Vergleich berechnen
NFFT = 2^12;  
Sx1x1 = abs(fft(h1,NFFT)).^2;                axis_Sx1=[0 1 0 1.35*max(Sx1x1)];
Sx2x2 = ones(1,NFFT)./abs(fft(h2,NFFT)).^2;  axis_Sx2=[0 1 0 1.35*max(Sx2x2)];
fT = 0:1/NFFT:1-1/NFFT;

% Schaetzung des Leistungsdichtespektrums fuer M=8, 32, 128
win = 'triang';

Sxx_x1_8   = lblack(x1,8,win,NFFT);
Sxx_x1_32  = lblack(x1,32,win,NFFT); 
Sxx_x1_128 = lblack(x1,128,win,NFFT);

Sxx_x2_8   = lblack(x2,8,win,NFFT);
Sxx_x2_32  = lblack(x2,32,win,NFFT); 
Sxx_x2_128 = lblack(x2,128,win,NFFT); 

% # grafische Ausgabe # %
figure;
plot(fT,Sx1x1,'--'); 
hold on;  
plot(fT,Sxx_x1_8); 
xlabel('\Omega/2\pi = f \cdot T'); 
ylabel('S_{XX}(exp(j\cdot \Omega))');
title('Bl.-Tukey-Sch. von S_{X1X1}(exp(j\cdot \Omega)), M=8 (Dreieck)');
axis(axis_Sx1);  

figure;   
plot(fT,Sx1x1,'--'); 
hold on;  
plot(fT,Sxx_x1_32); 
xlabel('\Omega/2\pi = f \cdot T'); 
ylabel('S_{XX}(exp(j\cdot \Omega))');
title('Bl.-Tukey-Sch. von S_{X1X1}(exp(j\cdot \Omega)), M=32 (Dreieck)'); 
axis(axis_Sx1); 
 
figure;  
plot(fT,Sx1x1,'--');
hold on; 
plot(fT,Sxx_x1_128);
xlabel('\Omega/2\pi = f \cdot T'); 
ylabel('S_{XX}(exp(j\cdot \Omega))');
title('Bl.-Tukey-Sch. von S_{X1X1}(exp(j\cdot \Omega)), M=128 (Dreieck)'); 
axis(axis_Sx1); 

input('....Zum Fortfahren: RETURN druecken');

figure;   
plot(fT,Sx2x2,'--'); 
hold on;  
plot(fT,Sxx_x2_8); 
xlabel('\Omega/2\pi = f \cdot T'); 
ylabel('S_{XX}(exp(j\cdot \Omega))');
title('Bl.-Tukey-Sch. von S_{X2X2}(exp(j\cdot \Omega)), M=8 (Dreieck)');
axis(axis_Sx2);  

figure;
plot(fT,Sx2x2,'--'); 
hold on; 
plot(fT,Sxx_x2_32); 
xlabel('\Omega/2\pi = f \cdot T'); 
ylabel('S_{XX}(exp(j\cdot \Omega))');
title('Bl.-Tukey-Sch. von S_{X2X2}(exp(j\cdot \Omega)), M=32 (Dreieck)'); 
axis(axis_Sx2); 

figure;  
plot(fT,Sx2x2,'--');
hold on; 
plot(fT,Sxx_x2_128); 
xlabel('\Omega/2\pi = f \cdot T'); 
ylabel('S_{XX}(exp(j\cdot \Omega))');
title('Bl.-Tukey-Sch. von S_{X2X2}(exp(j\cdot \Omega)), M=128 (Dreieck)'); 
axis(axis_Sx2); 
# END SOLUTION

### Lösung: Traditionelle Spektrumschätzung mit der Korrelationsmethode

Für verschiedene Werte von $M$ zeigen die folgenden Abbildungen eine Blackman–Tukey-Schätzung $\hat{S}{XX}^{BT}$ der Leistungsdichtespektren $S{XX}$ unter Verwendung von $N = 2^{12}$ Stichproben der Rauschprozesse $X_1(k)$ und $X_2(k)$. Es wird ein Dreiecksfenster der Länge $2M-1$ verwendet. Das echte Leistungsdichtespektrum $S_{\pi X_i \pi X_i}(\mathrm{e}^{j\Omega})$ ist als gestrichelte Linie dargestellt.

Aus der ersten Abbildung der ersten Figur ist ersichtlich, dass das echte Leistungsdichtespektrum für $M = 8$ gut approximiert wird.
Dies liegt daran, dass die echte Autokorrelationsfunktion $r_{XX}$ außerhalb des Intervalls $|\kappa| \leq m$ für ein MA-Modell der Ordnung $m = 4$ verschwindet.
In unserem Beispiel hat das MA-Modell $H_1(z)$ die Ordnung $m = 4$, sodass die echte ACF $r_{XX}$ auf das Intervall $|\kappa| \leq 4$ beschränkt ist.

Im Gegensatz dazu verschwindet die geschätzte ACF $\hat{r}{XX}$ außerhalb dieses Intervalls nicht, sondern nimmt nur kleine Werte an, abhängig von der Länge $N$ des Datenvektors.
Die Varianz der Schätzung kann verbessert werden, indem diese störenden ACF-Werte $\hat{r}{XX}$ mit $|\kappa| > 4$ durch Fensterung gedämpft oder vollständig unterdrückt werden.

Dies stellt den optimalen Fall für $M = 8$ dar, da das resultierende Fenster der Länge 15 alle ACF-Werte $\hat{r}_{XX}$ mit $|\kappa| > 7$ auf null setzt.
Die Multiplikation mit einem kurzen Fenster im Zeitbereich entspricht jedoch einer Faltung mit einem breiten Spektralfenster im Frequenzbereich, wodurch der Bias für kleine $M$-Werte größer wird als für größere Werte.

<img src="./pngs/ldsv8a03_x11.png" width="250">
<img src="./pngs/ldsv8a03_x12.png" width="250">
<img src="./pngs/ldsv8a03_x13.png" width="250">

Wir beobachten, dass der Bias mit steigenden Werten von $M$ abnimmt. Die Varianz nimmt jedoch proportional zu $\sum_\kappa f_M^2(\kappa)$ zu. Für das Dreiecksfenster gilt die Näherung:

$$
\sum_\kappa f_M^2(\kappa) \approx \frac{2}{3} M
$$

Dies impliziert, dass die Varianz proportional zu $M$ ist.

Im Gegensatz zum MA-Modell $H_1(z)$ hat das AR-Modell $H_2(z)$ eine unendlich lange Impulsantwort und daher eine unendlich ausgedehnte ACF $r_{XX}$. Die Fensterung verfälscht daher die geschätzten Leistungsdichtespektren in diesem Fall stärker.

<img src="./pngs/ldsv8a03_x21.png" width="250">
<img src="./pngs/ldsv8a03_x22.png" width="250">
<img src="./pngs/ldsv8a03_x23.png" width="250">

**Anmerkung:** Bei Verwendung eines Rechteckfensters treten für kleine Werte von $M$ starke Oszillationen bei niedrigen und hohen Frequenzen auf.